In [1]:
import sys
sys.path.append('/host/d/Github/')

import argparse
import os

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, confusion_matrix, roc_auc_score
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

import Osteosarcoma.Build_lists.Build_list as Build_list
import Osteosarcoma.functions_collection as ff


/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### define prediction result path from each fold for a particular fold

In [8]:
data_root = '/host/d/projects/Habitats/models/Prognosis/'
trial_name = 'resnet18_2D_FT1_roi'
fold0_results = pd.read_excel(os.path.join(data_root, trial_name, 'random0_fold0/predictions','prediction_epoch150_fold5.xlsx') )
fold1_results = pd.read_excel(os.path.join(data_root, trial_name, 'random0_fold1/predictions','prediction_epoch10_fold5.xlsx') )
fold2_results = pd.read_excel(os.path.join(data_root, trial_name, 'random0_fold2/predictions','prediction_epoch5_fold5.xlsx') )
fold3_results = pd.read_excel(os.path.join(data_root, trial_name, 'random0_fold3/predictions','prediction_epoch15_fold5.xlsx') )
fold4_results = pd.read_excel(os.path.join(data_root, trial_name, 'random0_fold4/predictions','prediction_epoch15_fold5.xlsx') )

In [9]:
# ============================================================
# Block 1. Merge five fold-model predictions into one dataframe
# ============================================================

from functools import reduce

common_columns = ["Patient_set", "Patient_index", "fold", "label"]

fold_result_list = [
    fold0_results,
    fold1_results,
    fold2_results,
    fold3_results,
    fold4_results,
]

fold_prediction_dfs = []

for fold_id, df in enumerate(fold_result_list):
    df_tmp = df.copy()

    required_columns = common_columns + ["probability"]
    missing_columns = [col for col in required_columns if col not in df_tmp.columns]
    if len(missing_columns) > 0:
        raise KeyError(f"fold{fold_id} missing columns: {missing_columns}")

    df_tmp = df_tmp[common_columns + ["probability"]].copy()
    df_tmp = df_tmp.rename(columns={"probability": f"fold{fold_id}"})

    fold_prediction_dfs.append(df_tmp)

merged_df = reduce(
    lambda left, right: pd.merge(
        left,
        right,
        on=common_columns,
        how="inner",
        validate="one_to_one",
    ),
    fold_prediction_dfs,
)

# Rename to your preferred lowercase style.
merged_df = merged_df.rename(
    columns={
        "Patient_set": "patient_set",
        "Patient_index": "patient_index",
    }
)

print("Merged dataframe shape:", merged_df.shape)
print(merged_df.head())

Merged dataframe shape: (98, 9)
  patient_set  patient_index  fold  label     fold0     fold1     fold2  \
0       set_1              5     5      1  0.325279  0.257817  0.217831   
1       set_1              8     5      1  0.017619  0.110440  0.210724   
2       set_1             15     5      0  0.207474  0.161923  0.309256   
3       set_1             20     5      0  0.028498  0.163834  0.339922   
4       set_1             28     5      0  0.836829  0.618236  0.095207   

      fold3     fold4  
0  0.287573  0.200737  
1  0.085085  0.077995  
2  0.183446  0.196746  
3  0.144997  0.117819  
4  0.629683  0.778014  


In [16]:
# ============================================================
# Block 2. Calculate fold_mean, best_fold, best_fold_p,
#          and fold_mean_indicated
# ============================================================

fold_prob_columns = ["fold0", "fold1", "fold2", "fold3", "fold4"]

# a. Mean probability across 5 fold-models
merged_df["fold_mean"] = merged_df[fold_prob_columns].mean(axis=1)


# b. AUC of each fold-model on this prediction dataset
fold_auc_dict = {}

for col in fold_prob_columns:
    auc = roc_auc_score(
        merged_df["label"].astype(int),
        merged_df[col].astype(float),
    )
    fold_auc_dict[col] = auc
    print(f"{col} AUC:", auc)

best_fold = max(fold_auc_dict, key=fold_auc_dict.get)
best_fold_auc = fold_auc_dict[best_fold]

print("\nBest fold:", best_fold)
print("Best fold AUC:", best_fold_auc)

merged_df["best_fold"] = best_fold
merged_df["best_fold_p"] = merged_df[best_fold]


# c. Cheating / indicated mean:
#    label=0 -> average the smallest 2 probabilities
#    label=1 -> average the largest 2 probabilities

def indicated_probability(row):
    probs = row[fold_prob_columns].astype(float).to_numpy()
    label = int(row["label"])

    probs_sorted = np.sort(probs)

    if label == 0:
        return float(np.mean(probs_sorted[:3]))
    elif label == 1:
        return float(np.mean(probs_sorted[-3:]))
    else:
        raise ValueError(f"Unexpected label: {label}")

merged_df["fold_mean_indicated"] = merged_df.apply(
    indicated_probability,
    axis=1,
)

print("\nFinal dataframe shape:", merged_df.shape)
print(merged_df.head())

fold0 AUC: 0.5627186406796602
fold1 AUC: 0.5557221389305348
fold2 AUC: 0.43778110944527737
fold3 AUC: 0.5292353823088456
fold4 AUC: 0.5527236381809095

Best fold: fold0
Best fold AUC: 0.5627186406796602

Final dataframe shape: (98, 13)
  patient_set  patient_index  fold  label     fold0     fold1     fold2  \
0       set_1              5     5      1  0.325279  0.257817  0.217831   
1       set_1              8     5      1  0.017619  0.110440  0.210724   
2       set_1             15     5      0  0.207474  0.161923  0.309256   
3       set_1             20     5      0  0.028498  0.163834  0.339922   
4       set_1             28     5      0  0.836829  0.618236  0.095207   

      fold3     fold4  fold_mean best_fold  best_fold_p  fold_mean_indicated  
0  0.287573  0.200737   0.257847     fold0     0.325279             0.290223  
1  0.085085  0.077995   0.100373     fold0     0.017619             0.135416  
2  0.183446  0.196746   0.211769     fold0     0.207474             0.180705

In [17]:
# ============================================================
# Block 3. Check AUCs for combined probabilities
# ============================================================

summary_auc = {
    "fold_mean": roc_auc_score(merged_df["label"], merged_df["fold_mean"]),
    "best_fold_p": roc_auc_score(merged_df["label"], merged_df["best_fold_p"]),
    "fold_mean_indicated": roc_auc_score(merged_df["label"], merged_df["fold_mean_indicated"]),
}

summary_auc_df = pd.DataFrame(
    [{"method": k, "AUC": v} for k, v in summary_auc.items()]
)

print(summary_auc_df)

                method       AUC
0            fold_mean  0.530735
1          best_fold_p  0.562719
2  fold_mean_indicated  0.817591
